# Importing Libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

# Loading Dataset

In [5]:
df=pd.read_csv('IMDb Movies India.csv',encoding='latin-1')
df.head()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali


# Inspecting Basic Info

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  object 
 1   Year      14981 non-null  object 
 2   Duration  7240 non-null   object 
 3   Genre     13632 non-null  object 
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   object 
 6   Director  14984 non-null  object 
 7   Actor 1   13892 non-null  object 
 8   Actor 2   13125 non-null  object 
 9   Actor 3   12365 non-null  object 
dtypes: float64(1), object(9)
memory usage: 1.2+ MB


In [7]:
df.shape

(15509, 10)

In [8]:
df.columns

Index(['Name', 'Year', 'Duration', 'Genre', 'Rating', 'Votes', 'Director',
       'Actor 1', 'Actor 2', 'Actor 3'],
      dtype='object')

# Data Preprocessing

In [10]:
if 'Name' in df.columns:
    df.drop(columns=['Name'], inplace=True)

In [11]:
df.dropna(subset=['Rating'], inplace=True)

In [12]:
if 'Year' in df.columns:
    df['Year'] = df['Year'].astype(str).str.extract(r'(\d+)').astype(float)

In [13]:
if 'Duration' in df.columns:
    df['Duration'] = df['Duration'].astype(str).str.extract(r'(\d+)').astype(float)

In [14]:
if 'Votes' in df.columns:
    df['Votes'] = df['Votes'].astype(str).str.replace(',', '').str.extract(r'(\d+)').astype(float)

In [15]:
num_cols = ['Year', 'Duration', 'Votes']
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

In [16]:
cat_cols = ['Genre', 'Director', 'Actor 1', 'Actor 2', 'Actor 3']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Feature Engineering

In [17]:
for col in cat_cols:
    if col in df.columns:
        mean_rating = df.groupby(col)['Rating'].transform('mean')
        df[f'{col}_encoded'] = mean_rating

In [18]:
df.drop(columns=[col for col in cat_cols if col in df.columns], inplace=True)

# Train-Test Split

In [19]:
X = df.drop(columns=['Rating'])
y = df['Rating']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

Training set size: 6335 samples
Testing set size: 1584 samples


# Model Training & Evaluation

In [21]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

In [22]:
for name, model in models.items():
    # Fit model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)

In [25]:
# Calculate Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\n[{name}] Performance:")
print(f"  Mean Absolute Error (MAE) : {mae:.4f}")
print(f"  Root Mean Squared Error(RMSE): {rmse:.4f}")
print(f"  R-squared (R2 Score)      : {r2:.4f}")


[Random Forest Regressor] Performance:
  Mean Absolute Error (MAE) : 0.4097
  Root Mean Squared Error(RMSE): 0.6127
  R-squared (R2 Score)      : 0.7981
